# Setup

In [ ]:
import os, sys, re, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

# try to install flash-attn from prebuilt wheels
torch_ver = re.match(r'(\d+\.\d+)', torch.__version__).group(1)
py_ver = f"{sys.version_info.major}{sys.version_info.minor}"

def _pip(url):
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", url], capture_output=True
    ).returncode == 0

installed = _pip(
    f"https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/"
    f"flash_attn-2.8.3+cu12torch{torch_ver}cxx11abiFALSE-cp{py_ver}-cp{py_ver}-linux_x86_64.whl"
)
if not installed:
    for v in ["2.8.3", "2.7.4", "2.6.3"]:
        if _pip(
            f"https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/"
            f"v0.7.15/flash_attn-{v}+cu126torch{torch_ver}-cp{py_ver}-cp{py_ver}"
            f"-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl"
        ):
            break

!pip install -qU transformers accelerate datasets peft trl bitsandbytes
!pip install -q anthropic

print(f"torch {torch.__version__} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0)}")
try:
    import flash_attn; print(f"flash_attn {flash_attn.__version__}")
except ImportError:
    print("no flash_attn, will use sdpa")

# Load data & split

In [ ]:
import json
from datasets import Dataset, DatasetDict

with open("cot_data.json") as f:
    cot_entries = json.load(f)
with open("combined_data.json") as f:
    distill_entries = json.load(f)

def split_data(entries, seed=42):
    """80/10/10 train/val/test split."""
    ds = Dataset.from_list(entries)
    s = ds.train_test_split(test_size=0.2, seed=seed)
    t = s["test"].train_test_split(test_size=0.5, seed=seed)
    return DatasetDict({"train": s["train"], "validation": t["train"], "test": t["test"]})

cot_splits = split_data(cot_entries)
distill_splits = split_data(distill_entries)

print(f"CoT:       {len(cot_splits['train'])} / {len(cot_splits['validation'])} / {len(cot_splits['test'])}")
print(f"Distilled: {len(distill_splits['train'])} / {len(distill_splits['validation'])} / {len(distill_splits['test'])}")

# Training & evaluation helpers

In [ ]:
import torch, gc, json, re
from tqdm import tqdm

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

try:
    import flash_attn; ATTN = "flash_attention_2"
except ImportError:
    ATTN = "sdpa"


def free():
    """Wipe model refs and flush GPU memory."""
    for name in ["model", "base_model", "trainer"]:
        if name in globals() and globals()[name] is not None:
            globals()[name] = None
    gc.collect()
    torch.cuda.empty_cache()


def load_model(quantize=False):
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    kw = {"device_map": {"": 0}, "attn_implementation": ATTN}
    if quantize:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        )
    else:
        kw["torch_dtype"] = torch.bfloat16

    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
    return mdl, tok


def _parse_groups(text, word_pool=None):
    """Try to pull groups of 4 words from model output -- JSON first, then line matching."""
    groups = []
    jm = re.search(r'\{.*?"answer"\s*:\s*\[.*?\]\s*\}', text, re.DOTALL)
    if jm:
        try:
            raw = [{w.strip().lower() for w in g} for g in json.loads(jm.group(0))["answer"]]
            groups = [g for g in raw if len(g) == 4]
            if word_pool:
                groups = [g for g in groups if g.issubset(word_pool)]
        except Exception:
            pass

    # fallback: scan lines for any that contain exactly 4 pool words
    if not groups and word_pool:
        for line in text.lower().split("\n"):
            found = {w for w in word_pool if re.search(r'\b' + re.escape(w) + r'\b', line)}
            if len(found) == 4 and found not in groups:
                groups.append(found)
    return groups


def _extract_targets(text):
    """Pull ground-truth groups from the training format."""
    m = re.search(r'So the answer is:\s*(\{.*?\})', text, re.DOTALL)
    if not m:
        return None
    try:
        return [{w.strip().lower() for w in g} for g in json.loads(m.group(1))["answer"]]
    except Exception:
        return None


def run_eval(model, tokenizer, test_data):
    correct = 0
    perfect = 0
    n = len(test_data)

    for item in tqdm(test_data, desc="eval"):
        text = item["text"]
        prompt = text.split("[/INST]")[0] + "[/INST]\n"

        targets = _extract_targets(text)
        if not targets:
            continue

        all_words = {w for g in targets for w in g}

        inp = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inp, max_new_tokens=400, temperature=0.1,
                pad_token_id=tokenizer.eos_token_id, do_sample=True
            )
        resp = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
        preds = _parse_groups(resp, all_words)

        hits = sum(1 for t in targets if t in preds)
        correct += hits
        if hits == 4:
            perfect += 1

    return correct, perfect, n


def run_gamified_eval(model, tokenizer, test_data, max_lives=4):
    """Simulate the actual Connections game: one guess at a time, lose a life on wrong guesses."""
    correct = 0
    perfect = 0
    n = len(test_data)

    pbar = tqdm(enumerate(test_data), total=n, desc="Gamified Eval")
    for idx, item in pbar:
        text = item["text"]
        base_inst = text.split("[/INST]")[0]

        targets = _extract_targets(text)
        if not targets:
            continue

        all_words = [w for g in targets for w in g]

        tqdm.write(f"\n{'='*40}\nPUZZLE {idx + 1}\n{'='*40}")

        lives = max_lives
        solved = []
        failed = []
        turn = 1
        temp = 0.1

        # main game loop
        while lives > 0 and len(solved) < 4:
            tqdm.write(f"\n--- Turn {turn} | Lives: {lives} | Temp: {temp:.1f} ---")

            solved_words = {w for g in solved for w in g}
            remaining = [w for w in all_words if w not in solved_words]

            # build prompt with game state context
            inst = base_inst
            if solved or failed:
                inst += "\n\nIMPORTANT GAME STATE UPDATE:\n"
                if solved:
                    inst += f"- Already found {len(solved)} groups. Remaining {len(remaining)} words: {remaining}\n"
                    inst += f"- The remaining words form exactly {4 - len(solved)} groups of 4.\n"
                if failed:
                    inst += f"- Wrong guesses so far: {[list(g) for g in failed]}. Do NOT repeat them.\n"
                    inst += "- Try a completely different grouping logic.\n"
                inst += "- Output your response strictly in the exact same JSON format as requested above."

            inp = tokenizer(inst + "\n[/INST]\n", return_tensors="pt").to("cuda")
            with torch.no_grad():
                out = model.generate(
                    **inp, max_new_tokens=400, temperature=temp,
                    pad_token_id=tokenizer.eos_token_id, do_sample=True
                )
            resp = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

            remaining_set = set(remaining)
            preds = _parse_groups(resp, remaining_set)
            # also filter out placeholder junk
            preds = [p for p in preds if not any("___" in w for w in p)]

            if not preds:
                tqdm.write("No valid groups parsed, bumping temp...")
                temp = min(0.9, temp + 0.3)
                turn += 1
                continue

            # pick first guess we haven't tried
            guess = next((p for p in preds if p not in solved and p not in failed), None)
            if not guess:
                tqdm.write("All guesses already tried, bumping temp...")
                temp = min(0.9, temp + 0.3)
                turn += 1
                continue

            tqdm.write(f"Guess: {list(guess)}")
            if guess in targets:
                solved.append(guess)
                tqdm.write("Correct!")
                temp = 0.1
            else:
                lives -= 1
                failed.append(guess)
                tqdm.write(f"Wrong. Lives left: {lives}")
                temp = min(0.9, temp + 0.15)

            turn += 1

        # out of lives -- give a few more tries to salvage remaining groups
        if len(solved) < 4:
            remaining_count = 4 - len(solved)
            tqdm.write(f"\nOut of lives! {remaining_count} groups remain, final guessing round...")

            for attempt in range(3):
                if len(solved) >= 4:
                    break

                solved_words = {w for g in solved for w in g}
                remaining = [w for w in all_words if w not in solved_words]
                groups_left = 4 - len(solved)

                final_inst = base_inst
                final_inst += f"\n\nYou must find exactly {groups_left} groups from these {len(remaining)} words: {remaining}"
                final_inst += f"\nEach group has exactly 4 words. Output ALL {groups_left} groups in the JSON format."

                inp = tokenizer(final_inst + "\n[/INST]\n", return_tensors="pt").to("cuda")
                with torch.no_grad():
                    out = model.generate(
                        **inp, max_new_tokens=400, temperature=0.1 + attempt * 0.3,
                        pad_token_id=tokenizer.eos_token_id, do_sample=True
                    )
                resp = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

                final_preds = _parse_groups(resp, set(remaining))
                got_new = False
                for p in final_preds:
                    if p not in solved:
                        tqdm.write(f"Final guess: {list(p)}")
                        if p in targets:
                            solved.append(p)
                            tqdm.write("Salvaged!")
                        else:
                            tqdm.write("Wrong.")
                        got_new = True

                if not got_new:
                    tqdm.write(f"Attempt {attempt+1} produced nothing new, retrying...")

            for t in targets:
                if t not in solved:
                    tqdm.write(f"Revealed: {list(t)}")

        correct += len(solved)
        if len(solved) == 4:
            tqdm.write("\nPERFECT!")
            perfect += 1
        else:
            tqdm.write(f"\nGAME OVER. {len(solved)}/4 groups.")

        done = idx + 1
        total_groups = done * 4
        tqdm.write(
            f"Running: {correct}/{total_groups} groups ({correct/total_groups*100:.1f}%) | "
            f"{perfect}/{done} perfect ({perfect/done*100:.1f}%)"
        )
        pbar.set_postfix(
            groups=f"{correct}/{total_groups} ({correct/total_groups*100:.0f}%)",
            perfect=f"{perfect}/{done} ({perfect/done*100:.0f}%)"
        )

    return correct, perfect, n


def show(label, correct, perfect, n):
    total = n * 4
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  Categories: {correct}/{total} ({correct/total*100:.1f}%)")
    print(f"  Perfect:    {perfect}/{n} ({perfect/n*100:.1f}%)")
    print(f"{'='*50}")


def dump_inference(model, tokenizer, test_data, filename):
    """Write model predictions vs ground truth to a text file for manual inspection."""
    with open(filename, "w") as f:
        for idx, item in enumerate(tqdm(test_data, desc="dumping")):
            text = item["text"]
            prompt = text.split("[/INST]")[0] + "[/INST]\n"
            gt = text.split("[/INST]")[-1].replace("</s>", "").strip()

            inp = tokenizer(prompt, return_tensors="pt").to("cuda")
            with torch.no_grad():
                out = model.generate(
                    **inp, max_new_tokens=400, temperature=0.1,
                    pad_token_id=tokenizer.eos_token_id, do_sample=True
                )
            resp = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

            f.write(f"=== PUZZLE {idx+1} ===\n")
            f.write(f"--- GROUND TRUTH ---\n{gt}\n\n")
            f.write(f"--- MODEL OUTPUT ---\n{resp}\n\n\n")
    print(f"saved to {filename}")


def finetune(model, tokenizer, splits, out_dir, epochs=3):
    from peft import LoraConfig, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig

    model = prepare_model_for_kbit_training(model)

    lora = LoraConfig(
        r=64, lora_alpha=128, lora_dropout=0.1, bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj", "lm_head"
        ]
    )

    args = SFTConfig(
        output_dir=out_dir, num_train_epochs=epochs,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        optim="adamw_torch_fused", learning_rate=2e-4,
        weight_decay=0.001, warmup_ratio=0.03, max_grad_norm=0.3,
        logging_steps=10, save_strategy="epoch",
        bf16=True, tf32=True, lr_scheduler_type="cosine",
        dataset_text_field="text", max_length=2048,
        packing=False, report_to="none"
    )

    trainer = SFTTrainer(
        model=model, train_dataset=splits["train"],
        eval_dataset=splits["validation"],
        peft_config=lora, processing_class=tokenizer, args=args
    )
    trainer.train()
    trainer.model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    return trainer

# Baseline (no finetuning)

In [ ]:
free()
model, tokenizer = load_model(quantize=False)
correct, perfect, n = run_eval(model, tokenizer, cot_splits["test"])
show("NON-FINETUNED BASELINE", correct, perfect, n)
dump_inference(model, tokenizer, cot_splits["test"], "baseline_inference.txt")
del model; free()

# CoT finetuning

In [ ]:
free()
model, tokenizer = load_model(quantize=True)

COT_DIR = "./mistral-cot"
trainer = finetune(model, tokenizer, cot_splits, COT_DIR)
del trainer, model; free()

In [ ]:
from peft import PeftModel

free()
base, tokenizer = load_model(quantize=False)
model = PeftModel.from_pretrained(base, COT_DIR)
model.eval()

correct, perfect, n = run_eval(model, tokenizer, cot_splits["test"])
show("FINETUNED -- CoT", correct, perfect, n)
dump_inference(model, tokenizer, cot_splits["test"], "cot_inference.txt")

del model; free()

# Distillation + CoT finetuning

In [ ]:
free()
model, tokenizer = load_model(quantize=True)

DISTILL_DIR = "./mistral-distill"
trainer = finetune(model, tokenizer, distill_splits, DISTILL_DIR)
del trainer, model; free()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/197 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/197 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss
10,0.976968
20,0.347573
30,0.304513
40,0.302305
50,0.289438
60,0.273589
70,0.272201
80,0.263135
90,0.263107
100,0.250582


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers fou

In [ ]:
from peft import PeftModel

free()
base, tokenizer = load_model(quantize=False)
model = PeftModel.from_pretrained(base, DISTILL_DIR).to("cuda")
model.eval()

dump_inference(model, tokenizer, distill_splits["test"], "distill_inference.txt")

del model; free()

In [ ]:
from peft import PeftModel

free()
base, tokenizer = load_model(quantize=False)
model = PeftModel.from_pretrained(base, DISTILL_DIR).to("cuda")
model.eval()

correct, perfect, n = run_gamified_eval(model, tokenizer, distill_splits["test"], max_lives=3)
show("FINETUNED -- Gamified Distillation (3 Lives)", correct, perfect, n)

del model; free()